# 05 Model Reliability: Splits, Leakage, and Applicability Domain

This section asks a more scientific question: when is a model trustworthy?

Application scenario: when a model ranks new molecules, screens candidates, or suggests experiments, a single random-split score is not enough. We need to know whether test molecules are merely nearest neighbors of training molecules, or whether they represent genuinely new chemical space.

## Intuition

If the test set contains many nearest neighbors of the training set, the model can look very good. A scaffold split is more like testing new exam questions, while a random split is more like changing numbers within the same question type.

This section uses Murcko scaffolds as an approximation of molecular core structures. The molecular-framework idea introduced by Bemis and Murcko is often used to group molecules by core scaffold. In machine-learning evaluation, scaffold split is a stricter structural-extrapolation check than random split.

## Mathematical and Chemical Definitions

Data leakage means that test information enters the training procedure. Applicability domain means the input region where model predictions have enough supporting evidence.

One simple applicability-domain proxy:

```text
nearest_train_similarity(test molecule) = max Tanimoto(test, each train molecule)
```

Lower similarity suggests that the model may be predicting out of distribution.

## Prepare Random Split and Scaffold Split

This cell builds the same ESOL feature table and then creates two splits:

- random split: randomly samples the test set and usually tests interpolation.
- scaffold split: groups by scaffold, so test-set core structures are more likely to be unseen in training.

In [ ]:
from pathlib import Path

START = Path.cwd().resolve()
for candidate in [START, *START.parents]:
    if (candidate / "data").exists() and (candidate / "notebooks").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Cannot find the materials root. Start Jupyter from the materials directory "
        "or from one of its subdirectories."
    )

DATA = ROOT / "data"
RAW = DATA / "raw"
EXAMPLES = DATA / "examples"
RANDOM_STATE = 42

print("materials root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

sns.set_theme(style="whitegrid", context="notebook")


def mol_from_smiles(smiles):
    if pd.isna(smiles):
        return None
    return Chem.MolFromSmiles(str(smiles).strip())


def canonical_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def scaffold_smiles(mol):
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)


def descriptor_record(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "canonical_smiles": canonical_smiles(smiles),
        "scaffold": scaffold_smiles(mol),
    }


def build_esol_features():
    raw = pd.read_csv(RAW / "esol.csv")
    rows = []
    for row_id, row in raw.reset_index(drop=True).iterrows():
        desc = descriptor_record(row["smiles"])
        if desc is None:
            continue
        desc.update(
            {
                "row_id": row_id,
                "smiles": str(row["smiles"]).strip(),
                "logS": float(row["log solubility (mol/L)"]),
            }
        )
        rows.append(desc)
    return pd.DataFrame(rows)


def fingerprint_array(smiles, n_bits=1024):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    matrix = np.zeros((len(smiles), n_bits), dtype=np.int8)
    for idx, smi in enumerate(smiles):
        fp = generator.GetFingerprint(mol_from_smiles(smi))
        DataStructs.ConvertToNumpyArray(fp, matrix[idx])
    return matrix


DESCRIPTOR_COLUMNS = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "AromaticRings",
    "FractionCSP3",
    "HeavyAtomCount",
]

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split

esol = build_esol_features()
X = esol[DESCRIPTOR_COLUMNS].to_numpy(dtype=float)
y = esol["logS"].to_numpy(dtype=float)
indices = np.arange(len(esol))

# random split ignores chemical scaffolds and samples randomly.
random_train, random_test = train_test_split(indices, test_size=0.2, random_state=RANDOM_STATE)

# GroupShuffleSplit keeps the same scaffold group from entering both train and test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
scaffold_train, scaffold_test = next(splitter.split(X, y, groups=esol["scaffold"]))

## Code

Use the same model to compare random split and scaffold split. The larger the difference, the more the evaluation depends on split strategy.

In [ ]:
def fit_and_score(train_idx, test_idx):
    # Keep the model fixed and change only the split, so differences mainly come from evaluation setup rather than model choice.
    model = RandomForestRegressor(n_estimators=60, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X[train_idx], y[train_idx])
    pred_train = model.predict(X[train_idx])
    pred_test = model.predict(X[test_idx])
    return {
        "train_RMSE": mean_squared_error(y[train_idx], pred_train) ** 0.5,
        "test_RMSE": mean_squared_error(y[test_idx], pred_test) ** 0.5,
        "pred_test": pred_test,
    }

scores = {
    "random split": fit_and_score(random_train, random_test),
    "scaffold split": fit_and_score(scaffold_train, scaffold_test),
}

pd.DataFrame(
    [
        {"split": name, "train_RMSE": value["train_RMSE"], "test_RMSE": value["test_RMSE"]}
        for name, value in scores.items()
    ]
).round(3)

## Estimate Applicability Domain with Nearest Training Neighbors

For each test molecule, compute its Morgan fingerprint Tanimoto similarity to every training molecule and keep the maximum value as `nearest_train_similarity`. This is not rigorous uncertainty, but it gives an intuitive applicability-domain proxy.

In [ ]:
def nearest_train_similarity(train_idx, test_idx):
    all_fp = []
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
    for smi in esol["smiles"]:
        all_fp.append(generator.GetFingerprint(mol_from_smiles(smi)))
    out = []
    train_fps = [all_fp[i] for i in train_idx]
    for idx in test_idx:
        # BulkTanimotoSimilarity computes similarities between the current test molecule and all training molecules at once.
        sims = DataStructs.BulkTanimotoSimilarity(all_fp[idx], train_fps)
        out.append(max(sims))
    return np.array(out)

random_nn = nearest_train_similarity(random_train, random_test)
scaffold_nn = nearest_train_similarity(scaffold_train, scaffold_test)

sim_df = pd.DataFrame(
    {
        "nearest_train_similarity": np.r_[random_nn, scaffold_nn],
        "split": ["random split"] * len(random_nn) + ["scaffold split"] * len(scaffold_nn),
    }
)

plt.figure(figsize=(7, 4))
sns.histplot(data=sim_df, x="nearest_train_similarity", hue="split", bins=25, element="step", stat="density", common_norm=False)
plt.title("Applicability domain proxy: nearest training similarity")
plt.tight_layout()

## Find Low-Similarity, High-Error Examples

This cell puts scaffold-split prediction error and nearest-neighbor similarity into one table. Ask whether these molecules are intrinsically difficult, or whether the training set lacks similar chemical environments.

In [ ]:
reliability_df = pd.DataFrame(
    {
        "true_logS": y[scaffold_test],
        "pred_logS": scores["scaffold split"]["pred_test"],
        "nearest_train_similarity": scaffold_nn,
        "smiles": esol.iloc[scaffold_test]["smiles"].to_numpy(),
    }
)
reliability_df["abs_error"] = (reliability_df["true_logS"] - reliability_df["pred_logS"]).abs()
reliability_df.sort_values(["nearest_train_similarity", "abs_error"], ascending=[True, False]).head(8).round(3)

## Visualize the Risk Relationship

Lower x-axis values mean the test molecule is farther from training molecules; higher y-axis values mean larger prediction error. If low-similarity regions show larger errors, model outputs should be treated as hypotheses, not certain conclusions.

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=reliability_df, x="nearest_train_similarity", y="abs_error", alpha=0.7)
plt.title("Low similarity often means higher risk")
plt.tight_layout()

## Observation Questions

1. Which test RMSE is higher: random split or scaffold split?
2. Are scaffold-split test molecules less similar to their nearest training neighbors?
3. How would you use a model prediction for a molecule with very low nearest similarity?

### Hints

1. Scaffold split is often higher, but the exact result depends on seed and data distribution. The key is explaining why.
2. If scaffold split successfully creates structural extrapolation, the nearest-neighbor similarity distribution should shift lower.
3. Low-similarity predictions are better treated as hypotheses requiring verification. You can request experiments, search for external similar data, or let the model abstain.

## Summary

Model reliability is not a single score. Examine splits, duplicates, nearest-neighbor similarity, failure cases, and chemical explanations together. For low-similarity samples, predictions should be treated as hypotheses rather than conclusions.